In [1]:
import requests
import json
import re
from html import unescape

"""
과제: 네이버 블로그 검색 OpenAPI를 활용한 데이터 수집 파이프라인

참고 사이트
- 네이버 블로그 검색 OpenAPI
  https://developers.naver.com/docs/serviceapi/search/blog/blog.md

목표
- 네이버 블로그 검색 OpenAPI를 호출하여 데이터를 수집한다.
- 페이지네이션(start 파라미터)을 사용해 여러 번 API를 호출한다.
- 수집한 데이터를 하나의 리스트로 누적한다.
- JSON 파일로 저장한다.
"""

'\n과제: 네이버 블로그 검색 OpenAPI를 활용한 데이터 수집 파이프라인\n\n참고 사이트\n- 네이버 블로그 검색 OpenAPI\n  https://developers.naver.com/docs/serviceapi/search/blog/blog.md\n\n목표\n- 네이버 블로그 검색 OpenAPI를 호출하여 데이터를 수집한다.\n- 페이지네이션(start 파라미터)을 사용해 여러 번 API를 호출한다.\n- 수집한 데이터를 하나의 리스트로 누적한다.\n- JSON 파일로 저장한다.\n'

In [2]:
# 네이버 개발자 센터에서 발급받은 인증 정보 입력
CLIENT_ID = "EDZj4GtOZnrUuOC5quUq"
CLIENT_SECRET = "EV09c2U784"

# 블로그 검색 OpenAPI 엔드포인트
URL = "https://openapi.naver.com/v1/search/blog.json"

# 문제 1
# OpenAPI 호출에 필요한 인증 정보를 헤더에 설정하시오.
HEADERS = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET,
}

In [3]:
# 문제 2
# 검색어(query) : 인공지능, 한 번에 가져올 개수(display) 10 , 시작 위치(start) 1를 설정하시오.
PARAMS = {
    "query": "인공지능",
    "display": 5,
    "start": 1,
}


In [4]:
# HTML 태그 제거 함수 (수정 금지)
def clean_html(text):
    if text is None:
        return ""
    text = unescape(text)
    return re.sub(r"<.*?>", "", text)

In [5]:
# 문제 3
# 여러 번의 API 호출 결과를 저장할 빈 리스트를 생성하시오.
all_items = []

# 문제 4
# start 값을 1, 11, 21로 변경하면서 API를 3번 호출하시오.
for start in [1,11,21]:
    PARAMS["start"] = start

    # 문제 5
    # GET 요청을 보내고 응답을 받으시오.
    resp = requests.get(url=URL, headers=HEADERS, params=PARAMS)

    print("status:", resp.status_code, "start:", start)

    # 문제 6
    # 상태 코드가 200이 아닐 경우 에러를 발생시키시오.
    if resp.status_code != 200:
        print(resp.text)
        raise RuntimeError("API 호출 실패")

    # 문제 7
    # JSON 응답에서 items 데이터를 추출하여 all_items에 누적하시오.
    data = resp.json()
    items = data['items']
    all_items.extend(items)


status: 200 start: 1
status: 200 start: 11
status: 200 start: 21


In [6]:
# 문제 8
# 수집된 원본 데이터를 정제하여 results 리스트를 완성하시오.
results = []
for it in all_items:
    results.append({
        "title": clean_html(it.get("title")),
        "description": clean_html(it.get("description")),
        "link": it.get("link"),
        "blogger": it.get("bloggername"),
        "postdate": it.get("postdate"),
    })


In [8]:
# 문제 9
# results 데이터를 JSON 파일로 저장하시오.
with open("naver_blog_pipeline.json", "w", encoding="utf-8") as f:
    json.dump(results,f,ensure_ascii=False, indent=4)

print("저장 완료")
print("수집 건수:", len(results))

저장 완료
수집 건수: 15
